## Sanity-check of the method

As a first test, we will stablish the workflow of the dynamic multi-product formulas (DMPF) on the same Heisenberg Hamiltonian given in equation (12) of Niall's paper for a small number of spins.

### Computing the Gram matrix elements and the overlap vector

We will check these results by computing $\left< \psi \right| S^{\dagger} \left( \frac{t}{k_i} \right)^{k_i} S \left( \frac{t}{k_j} \right)^{k_j} \left| \psi \right>$ in two different ways:
1. Compute the MPO $F_{ij} = S^{\dagger} \left( \frac{t}{k_i} \right)^{k_i} S \left( \frac{t}{k_j} \right)^{k_j}$ using middle-out contraction. Represent the state $\left| \psi \right>$ by an MPS. Compute the contraction of the MPO with the MPS (left and right).
2. Represent the state $\left| \psi \right>$ by an MPS. Apply $S \left( \frac{t}{k_j} \right)^{k_j}$ as you would normallly do (contracting each layer with the MPS, generating a new MPS every step of the way OR (approach used here) build first the MPO and then apply it to the state). Do the same for the state $\left< \psi \right|$ and the circuit $S^{\dagger} \left( \frac{t}{k_i} \right)^{k_i}$. Compute the contraction between the two resulting MPS.

In [1]:
include("utils.jl")
using Distributions
using Random

n = 3

Random.seed!(1234)
J =  rand(Uniform(1/4, 3/4), n-1)

t = 3
k = [3,8]
cutoff = 0
maxdim = 1000
sites = siteinds("Qubit", n)
initial_state = ["1", "0", "1"]

3-element Vector{String}:
 "1"
 "0"
 "1"

FIRST METHOD

In [ ]:
F_comp = build_F(n, J, t, k, sites, cutoff, maxdim)
psi = MPS(sites, initial_state)
normalize!(psi)

evs = [inner(psi', F, psi) for F in F_comp]
print(evs)

Any[0.9999999999999919, 0.8554740322379081, 0.8558943500572844, 0.9999999999999888]

SECOND METHOD

In [3]:
evs = []
for i in eachindex(k)
    S_i = get_MPO(n, J, t, k[i], sites, cutoff, maxdim)
    for j in eachindex(k)
        S_j = get_MPO(n, J, t, k[j], sites, cutoff, maxdim)

        psi0_i = MPS(sites, initial_state)
        psi0_j = MPS(sites, initial_state)

        psi_i = apply(S_i, psi0_i; cutoff=cutoff, maxdim=maxdim)
        psi_j = apply(S_j, psi0_j; cutoff=cutoff, maxdim=maxdim)

        a = real( inner(psi_i, psi_j) )
        push!(evs, a)
    end
end
print(evs)

Any[0.9999999999999948, 0.9683099246344452, 0.9683099246344453, 0.999999999999966]

Both methods yield the exact same results and they also make sense: we are computing $\left< \psi \right| F_{ij} \left| \psi \right>$, with $i,j = 1,2$. The diagonal elements of this "matrix", namely $F_{11}$ and $F_{22}$ correspond exactly to the identity operator (we are forward and backwards propagating by the Trotterized circuit the same amount of times on each direction), so the expectation value for any normalized state $\left|\psi \right>$ should yield 1. On the other hand, off-diagonal elements like $F_{12}$ and $F_{21}$ correspond to propagating unevenly in time (assuming $k_i \neq k_j$), which results in something close to (but not exactly) the identity operator.

Another thing pending to do is the interpretation of each element of the MPOs $F_{ij}$.

In [3]:
for element in F_comp
    print(element.data)
end

ITensor[ITensor ord=3
Dim 1: (dim=2|id=337|"Qubit,Site,n=1")'
Dim 2: (dim=2|id=337|"Qubit,Site,n=1")
Dim 3: (dim=4|id=980|"Link,l=1")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2×4
[:, :, 1] =
     -2.0000000000000178 + 2.569059257801529e-36im  2.0755245855328228e-27 - 9.321580665476716e-24im
 -3.5542490597466236e-22 - 4.208441875055254e-24im     -1.9999999999999902 - 6.205009367927508e-25im

[:, :, 2] =
  -4.840563669763393e-15 - 4.982140578458303e-15im   2.0537035197474474e-16 - 1.3213196499124505e-16im
 -1.0799055269208098e-16 + 1.4329662007287874e-16im   4.840563669763459e-15 + 4.982140578458371e-15im

[:, :, 3] =
   3.70388205818186e-18 + 1.0349511940873397e-17im   4.425868516762766e-16 + 1.8684692859036118e-16im
 -9.472109594561239e-17 + 4.3649634901417676e-16im  -3.703882058181911e-18 - 1.0349511940873538e-17im

[:, :, 4] =
 -1.008709223001384e-18 - 1.4021992128833829e-18im  -3.394144391117315e-17 + 7.71283928717483e-17im
  8.827871572197986e-17 + 2.0639775439086375e-17i

For this, we will contrast the tensor elements of the MPOs with the matrices obtained by computing $F_{ij}$ directly through matrix multiplication.

In [4]:
F_matrix = matrix_F(num_emitters, omega_m, omega_c, g, gamma, kappa, t, k)
using PrettyTables
for element in F_matrix
    pretty_table(real(round.(Matrix(element), digits=4)))
end

┌────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┐
│ Col. 1 │ Col. 2 │ Col. 3 │ Col. 4 │ Col. 5 │ Col. 6 │ Col. 7 │ Col. 8 │
├────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│    1.0 │    0.0 │    0.0 │    0.0 │    0.0 │    0.0 │    0.0 │    0.0 │
│    0.0 │    1.0 │    0.0 │    0.0 │    0.0 │    0.0 │    0.0 │    0.0 │
│    0.0 │    0.0 │    1.0 │    0.0 │    0.0 │    0.0 │    0.0 │    0.0 │
│    0.0 │    0.0 │    0.0 │    1.0 │    0.0 │    0.0 │    0.0 │    0.0 │
│    0.0 │    0.0 │    0.0 │    0.0 │    1.0 │    0.0 │    0.0 │    0.0 │
│    0.0 │    0.0 │    0.0 │    0.0 │    0.0 │    1.0 │    0.0 │    0.0 │
│    0.0 │    0.0 │    0.0 │    0.0 │    0.0 │    0.0 │    1.0 │    0.0 │
│    0.0 │    0.0 │    0.0 │    0.0 │    0.0 │    0.0 │    0.0 │    1.0 │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┘
┌────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┐
│ Col. 1 │ Col. 2 │ Col. 3 │ Col. 4 │ 